<a href="https://colab.research.google.com/github/aronnaiqbal/220153_CNN-Image-Classification/blob/main/220153_CNN_Image_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/aronnaiqbal/220153_CNN-Image-Classification.git

In [ ]:
import os

for root, dirs, files in os.walk('/content/220153_CNN-Image-Classification'):
    print(root)

In [ ]:
!wget https://storage.googleapis.com/download.tensorflow.org/data/rps.zip
!wget https://storage.googleapis.com/download.tensorflow.org/data/rps-test-set.zip

!unzip -q rps.zip
!unzip -q rps-test-set.zip

--2026-08-16 23:12:12--  https://storage.googleapis.com/download.tensorflow.org/data/rps.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 172.217.214.207, 108.177.121.207, 142.250.152.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.217.214.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 200682221 (191M) [application/zip]
Saving to: ‘rps.zip.1’

rps.zip.1           100%[===================>] 191.38M   134MB/s    in 1.4s    

2026-08-16 23:12:14 (134 MB/s) - ‘rps.zip.1’ saved [200682221/200682221]

--2026-08-16 23:12:14--  https://storage.googleapis.com/download.tensorflow.org/data/rps-test-set.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 172.217.214.207, 108.177.121.207, 142.250.152.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.217.214.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 29516758 (28M) [application/zip]
Saving to: ‘rps-test-s

In [ ]:
import os

print("Classes in training dataset:")
print(os.listdir("/content/rps"))

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [ ]:
train_dataset = ImageFolder(
    root="/content/rps",
    transform=transform
)

test_dataset = ImageFolder(
    root="/content/rps-test-set",
    transform=transform
)

print("Training Images:", len(train_dataset))
print("Test Images:", len(test_dataset))
print("Classes:", train_dataset.classes)

In [ ]:
from collections import Counter

labels = [label for _, label in train_dataset]

class_counts = Counter(labels)

for idx, count in sorted(class_counts.items()):
    print(f"{train_dataset.classes[idx]} : {count}")

In [ ]:
from collections import Counter

labels = [label for _, label in train_dataset]

class_counts = Counter(labels)

for idx, count in sorted(class_counts.items()):
    print(f"{train_dataset.classes[idx]} : {count}")

In [ ]:
import matplotlib.pyplot as plt

class_names = []
counts = []

for idx, count in sorted(class_counts.items()):
    class_names.append(train_dataset.classes[idx])
    counts.append(count)

plt.figure(figsize=(6,4))

plt.bar(class_names, counts)

plt.title("Class Distribution")
plt.xlabel("Classes")
plt.ylabel("Number of Images")

plt.show()

In [ ]:
plt.figure(figsize=(10,8))

for i in range(9):

    image, label = train_dataset[i]

    image = image.permute(1, 2, 0)

    image = (image * 0.5) + 0.5

    plt.subplot(3,3,i+1)

    plt.imshow(image)

    plt.title(train_dataset.classes[label])

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
image, label = train_dataset[0]

print("Image Shape:", image.shape)
print("Class:", train_dataset.classes[label])

In [ ]:
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_data, val_data = random_split(
    train_dataset,
    [train_size, val_size]
)

print("Train Samples:", len(train_data))
print("Validation Samples:", len(val_data))

In [ ]:
batch_size = 64

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("DataLoaders Ready")

In [ ]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, 3)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = torch.flatten(x, 1)
        x = self.fc_layers(x)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(device)

In [ ]:
train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

print("History Lists Ready")


In [ ]:
epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total

    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.2f}%"
    )

In [ ]:
torch.save(
    model.state_dict(),
    "220153.pth"
)

print("Model Saved Successfully")

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs Epoch")

plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy vs Epoch")

plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=train_dataset.classes,
    yticklabels=train_dataset.classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
import random

wrong_samples = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        for i in range(len(labels)):

            if preds[i].cpu() != labels[i]:

                wrong_samples.append(
                    (
                        images[i].cpu(),
                        labels[i].item(),
                        preds[i].cpu().item()
                    )
                )

num_show = min(3, len(wrong_samples))

selected = random.sample(wrong_samples, num_show)

plt.figure(figsize=(12,4))

for i, (img, true_label, pred_label) in enumerate(selected):

    plt.subplot(1, num_show, i + 1)

    img = img.permute(1, 2, 0).numpy()
    img = (img * 0.5) + 0.5

    plt.imshow(img)

    plt.title(
        f"True: {train_dataset.classes[true_label]}\nPred: {train_dataset.classes[pred_label]}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os

model_dir = "/content/220153_CNN-Image-Classification/model"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "220153.pth")

torch.save(model.state_dict(), model_path)

print("Model saved:", model_path)


In [ ]:
!rm -rf /content/220153_CNN-Image-Classification
!git clone https://github.com/aronnaiqbal/220153_CNN-Image-Classification.git

In [ ]:
import os

custom_path = "/content/220153_CNN-Image-Classification/dataset/custom_images"

print(os.listdir(custom_path))

In [ ]:
from PIL import Image
import os

custom_path = "/content/220153_CNN-Image-Classification/dataset/custom_images"

for filename in sorted(os.listdir(custom_path)):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        path = os.path.join(custom_path, filename)
        try:
            img = Image.open(path)
            print(filename, "✓", img.size, img.format)
        except:
            print(filename, "✗ Invalid image")

In [ ]:
from PIL import Image
import os
import torch
import matplotlib.pyplot as plt

custom_path = "/content/220153_CNN-Image-Classification/dataset/custom_images"

image_files = sorted([
    f for f in os.listdir(custom_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

class_names = train_dataset.classes

model.eval()

plt.figure(figsize=(15, 12))

for i, image_name in enumerate(image_files):

    image_path = os.path.join(custom_path, image_name)

    image = Image.open(image_path).convert("RGB")

    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.softmax(output, dim=1)
        confidence, prediction = torch.max(probabilities, dim=1)

    predicted_class = class_names[prediction.item()]
    confidence_percent = confidence.item() * 100

    plt.subplot(4, 3, i + 1)
    plt.imshow(image)

    plt.title(
        f"{image_name}\nPred: {predicted_class} ({confidence_percent:.1f}%)"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os

model_dir = "/content/220153_CNN-Image-Classification/model"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "220153.pth")

torch.save(model.state_dict(), model_path)

print("Model saved:", model_path)

In [ ]:
from PIL import Image
import os
import pandas as pd

custom_path = "/content/220153_CNN-Image-Classification/dataset/custom_images"

data = []

for img_name in sorted(os.listdir(custom_path)):
    if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        img_path = os.path.join(custom_path, img_name)

        try:
            img = Image.open(img_path)

            data.append([
                img_name,
                img.format,
                img.size[0],
                img.size[1],
                img.mode
            ])

        except:
            pass

df = pd.DataFrame(
    data,
    columns=["Filename", "Format", "Width", "Height", "Color Mode"]
)

df

In [ ]:
df.describe()